# Steinmetz SPN clustering

Download the Steinmetz et al. (2019) recordings from:

- Publication: https://www.nature.com/articles/s41586-019-1787-x
- Official dataset: https://figshare.com/articles/dataset/Distributed_coding_of_choice_action_and_engagement_across_the_mouse_brain/9974357

Place these session folders directly under `data/source/steinmetz/`:
```text
data/source/steinmetz/
├── session_manifest.csv
├── Hench_2017-06-18/
├── Lederberg_2017-12-11/
├── Radnitz_2017-01-12/
└── Richards_2017-11-01/
```

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import numpy as np
import pandas as pd

from spn_figures.config import DERIVED, SOURCE
from spn_figures.io import read_csv, write_csv
from spn_figures.datasets import load_steinmetz_session_exact
from spn_figures.clustering import (
    cluster_steinmetz_session_exact,
    steinmetz_plot_tables_exact,
    steinmetz_population_activity_table_exact,
)

source_dir = SOURCE / "steinmetz"
manifest = read_csv(source_dir / "session_manifest.csv")
out_dir = DERIVED / "steinmetz"
out_dir.mkdir(parents=True, exist_ok=True)

manifest = read_csv(source_dir / "session_manifest.csv")

print(f"Loaded {len(manifest)} manuscript sessions.")

Loaded 4 manuscript sessions.


In [3]:
profiles = []
correlations = []
activities = []
labels = []
selected_trials = []
summaries = []

for row in manifest.itertuples(index=False):
    payload = load_steinmetz_session_exact(
        source_dir / row.folder,
        session_name=row.session,
        matching_seed=int(row.matching_seed),
        premove_gap_s=float(row.premove_gap_s),
    )
    result = cluster_steinmetz_session_exact(payload)
    profile_table, correlation_table, plot_metadata = steinmetz_plot_tables_exact(result, plot_matching_seed=27)

    profiles.append(profile_table)
    correlations.append(correlation_table)
    activities.append(steinmetz_population_activity_table_exact(result))
    labels.append(
        pd.DataFrame(
            {
                "session": result["session"],
                "unit_id": result["unit_ids"],
                "lr_pref": result["lr_pref"],
                "final_label": result["final_labels"],
                "final_name": result["final_names"],
            }
        )
    )

    flat_info = result["naming_trial_info"]
    selected = flat_info["trials_all"]
    selected_trials.append(
        pd.DataFrame(
            {
                "session": result["session"],
                "trial_id": selected,
                "choice_left": (
                    payload["choice"][selected] == payload["choice_left_code"]
                ).astype(int),
                "decision_time_ms": 1000.0 * payload["reaction_time_s"][selected],
                "signed_contrast": payload["signed_contrast_all"][selected],
                "evidence_magnitude": payload["evidence_magnitude_all"][selected],
                "matching_seed": int(payload["matching_seed"]),
            }
        )
    )

    counts = pd.Series(result["final_names"]).value_counts()
    summaries.append(
        {
            "session": result["session"],
            "matching_seed": int(payload["matching_seed"]),
            "premove_gap_s": float(payload["premove_gap_s"]),
            "n_plot_trials": int(plot_metadata["n_plot_trials"]),
            "n_units_after_original_filters": int(len(result["unit_ids"])),
            "n_units_assigned": int(np.sum(result["final_labels"] >= 0)),
            "n_dSPN_left": int(counts.get("dSPN_left", 0)),
            "n_iSPN_left": int(counts.get("iSPN_left", 0)),
            "n_dSPN_right": int(counts.get("dSPN_right", 0)),
            "n_iSPN_right": int(counts.get("iSPN_right", 0)),
        }
    )

profiles = pd.concat(profiles, ignore_index=True)
correlations = pd.concat(correlations, ignore_index=True)
activities = pd.concat(activities, ignore_index=True)
labels = pd.concat(labels, ignore_index=True)
selected_trials = pd.concat(selected_trials, ignore_index=True)
summary = pd.DataFrame(summaries)

profiles.to_csv(out_dir / "unit_profiles.csv.gz", index=False)
write_csv(correlations, out_dir / "correlations.csv")
activities.to_csv(out_dir / "population_activity_bins.csv.gz", index=False)
write_csv(labels, out_dir / "unit_labels.csv")
write_csv(selected_trials, out_dir / "selected_trials.csv")
write_csv(summary, out_dir / "session_summary.csv")

summary

/Users/zhuojunyu/miniconda3/envs/ibl/lib/python3.10/site-packages/scipy/stats/_wilcoxon.py:172: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


,session,matching_seed,premove_gap_s,n_plot_trials,n_units_after_original_filters,n_units_assigned,n_dSPN_left,n_iSPN_left,n_dSPN_right,n_iSPN_right
0,Hench_2017-06-18,2,0.01,146,53,50,3,19,24,4
1,Lederberg_2017-12-11,4,0.01,148,31,29,3,13,9,4
2,Radnitz_2017-01-12,5,0.01,24,50,50,7,15,25,3
3,Richards_2017-11-01,27,0.01,68,35,35,5,7,10,13
